# **2. Генерация признаков**

После предварительного анализа переходим уже к построению самих признаков для обучения модели.

**Цель:** превратить сырые события в таблицу признаков (одна строка = одна cookie_id).

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data")

## **2.1. Загрузка данных**

In [56]:
train = pd.read_csv(DATA_DIR / "train.csv")
for col in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
    train[col] = pd.to_datetime(train[col])

test = pd.read_csv(DATA_DIR / "test.csv")
for col in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
    test[col] = pd.to_datetime(test[col])

events = pd.read_csv(DATA_DIR / "events.csv")
events["event_ts"] = pd.to_datetime(events["event_ts"])

print(f"\nTrain: {train.shape}")
print(f"Test: {test.shape}")
print(f"Events: {events.shape}")


Train: (11091, 5)
Test: (4909, 4)
Events: (328905, 14)


Получим объединенную таблицу трейна и событий с фильтрацией по временному окну.

In [57]:
def filter_events_by_window(events_df, targets_df):
    """
    Фильтрует события, оставляя только те, что попали в окно наблюдения.
    Возвращает events + target (если есть).
    """
    # Берем нужные колонки из targets
    cols = ["cookie_id", "window_start_ts", "window_end_ts"]
    if "target" in targets_df.columns:
        cols.append("target")
    
    # Merge и фильтрация
    merged = events_df.merge(targets_df[cols], on="cookie_id", how="inner")
    mask = (
        (merged["event_ts"] >= merged["window_start_ts"]) & 
        (merged["event_ts"] <= merged["window_end_ts"])
    )
    filtered = merged.loc[mask].copy()
    
    # Удаляем служебные колонки окна
    filtered = filtered.drop(columns=["window_start_ts", "window_end_ts"])
    return filtered

In [58]:
events_train = filter_events_by_window(events, train)
print(f"Событий в окне (train): {len(events_train)}")

events_test = filter_events_by_window(events, test)
print(f"Событий в окне (test): {len(events_test)}")

Событий в окне (train): 198504
Событий в окне (test): 89690


## **2.2. Генерация признаков**

Собираем все инсайты из EDA в один набор данных.

### **2.2.1. Базовые счетчики**

Гипотезы:

* Боты делают больше событий за 24 часа (парсят активно)

* Боты смотрят больше уникальных товаров (unique_items выше)

* Боты фокусируются на узкой нише (unique_categories ниже)

In [59]:
base_features = events_train.groupby("cookie_id").agg(
    event_count=("event_name", "size"),             # общее число событий
    unique_items=("item_id", "nunique"),            # уникальных товаров
    unique_categories=("item_category", "nunique"), # уникальных категорий
    unique_locations=("item_location", "nunique"),  # уникальных локаций
    unique_event_types=("event_name", "nunique"),   # уникальных типов событий
).reset_index()

# Объединяем с target для train
base_features = base_features.merge(
    train[["cookie_id", "target"]], on="cookie_id", how="left"
)

print(f"Shape: {base_features.shape}")
display(base_features.head())

Shape: (11091, 7)


,cookie_id,event_count,unique_items,unique_categories,unique_locations,unique_event_types,target
0,ck_000c95f1408dcb00,16,9,2,8,4,0
1,ck_000e8c52636e3bec,34,20,4,12,7,0
2,ck_0010e31baa4a1fb7,16,8,3,4,6,0
3,ck_0010ec3874fb5378,74,31,2,19,6,0
4,ck_001722063b94cae0,15,8,3,9,4,0


### **2.2.2. user_agent признаки**

Гипотеза: боты часто используют headless-браузеры, скрипты на Python/curl, имеют короткие или стандартные User-Agent строки. Люди используют обычные браузеры с длинными UA.

Признаки:

- `ua_length` - медианная длина строки User-Agent (короткая = бот)

- `has_headless` - флаг HeadlessChrome

- `has_bot_pattern` - флаг bot-паттернов

- `n_unique_ua` - количество уникальных UA

In [60]:
import re

# Предобработка
ua = events_train[['cookie_id', 'user_agent']].copy()
ua['ua_clean'] = ua['user_agent'].fillna('').str.lower()
ua['ua_len'] = ua['ua_clean'].str.len()

# флаг наличия бот-паттернов и флаг headless
BOT_PATTERNS = re.compile(r'bot|crawler|spider|python|curl|wget|scrapy|selenium')
ua['is_headless'] = ua['ua_clean'].str.contains('headless', regex=False)
ua['is_bot'] = ua['ua_clean'].str.contains(BOT_PATTERNS)

# Аагрегация
ua_features = ua.groupby('cookie_id').agg(
    ua_length=('ua_len', 'median'),
    has_headless=('is_headless', 'any'),
    has_bot_pattern=('is_bot', 'any'),
    n_unique_ua=('user_agent', 'nunique'),
).reset_index()

ua_features[['has_headless', 'has_bot_pattern']] = \
    ua_features[['has_headless', 'has_bot_pattern']].astype(int)

base_features = base_features.merge(ua_features, on='cookie_id', how='left')

### **2.2.3. Признаки платформы**

Гипотеза: боты доминируют на Web-платформе, тогда как люди более равномерно распределены между мобильными и веб-платформами.

In [61]:
# Предобработка: приводим к единому формату (как в EDA)
events_train['platform_clean'] = events_train['platform'].str.lower()
events_train['platform_clean'] = events_train['platform_clean'].replace({'desktop': 'web', 'iphone': 'ios'})

# Считаем долю каждой платформы для каждой куки
platform_counts = (
    events_train.groupby('cookie_id')['platform_clean']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

platform_counts = platform_counts.rename(columns={
    'web': 'platform_web_ratio',
    'android': 'platform_android_ratio',
    'ios': 'platform_ios_ratio'
})

# Если какой-то платформы не было у куки — заполняем нулем
for col in ['platform_web_ratio', 'platform_android_ratio', 'platform_ios_ratio']:
    if col not in platform_counts.columns:
        platform_counts[col] = 0.0

# Объединяем с предыдущими признаками
base_features = base_features.merge(
    platform_counts[['cookie_id', 'platform_web_ratio', 'platform_android_ratio', 'platform_ios_ratio']],
    on='cookie_id', how='left'
)

**Что получили:**

- `platform_web_ratio` - доля событий с Web

- `platform_android_ratio` - доля событий с Android

- `platform_ios_ratio` - доля событий с iOS

### **2.2.4. Признаки из seller_type**

Гипотеза: боты могут фокусироваться на определенных типах продавцов (например, только профессионалы), тогда как люди более разнообразны в выборе. Также отсутствие информации о продавце (unknown) может указывать на события поиска, где боты активнее.

In [62]:
# Заполняем пропуски значением 'unknown'
events_train['seller_type_clean'] = events_train['seller_type'].fillna('unknown')

# Считаем долю каждого типа продавца для каждой куки
seller_counts = (
    events_train.groupby('cookie_id')['seller_type_clean']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

# Переименовываем колонки
seller_counts = seller_counts.rename(columns={
    'private': 'seller_private_ratio',
    'pro': 'seller_pro_ratio',
    'unknown': 'seller_unknown_ratio'
})

# Если какого-то типа не было у куки — заполняем нулем
for col in ['seller_private_ratio', 'seller_pro_ratio', 'seller_unknown_ratio']:
    if col not in seller_counts.columns:
        seller_counts[col] = 0.0

# Объединяем с предыдущими признаками
base_features = base_features.merge(
    seller_counts[['cookie_id', 'seller_private_ratio', 'seller_pro_ratio', 'seller_unknown_ratio']],
    on='cookie_id', how='left'
)

**Что получили:**

- `seller_private_ratio` - доля событий с частными продавцами

- `seller_pro_ratio` - доля событий с профессиональными продавцами

- `seller_unknown_ratio` - доля событий без информации о продавце

### **2.2.5. Временные признаки**

Гипотезы:

- Возраст куки: Боты часто создают "свежие" куки специально для парсинга. Люди обычно имеют куки, которые существуют месяцами или годами.

- Ночная активность: Люди спят ночью, боты работают 24/7.

- Скорость действий: Боты делают события быстрее (короткие интервалы).

Признаки возраста куки:

- `is_created_in_window` - флаг: кука создана внутри окна

- `cookie_age_log` - логарифм возраста (для сглаживания выбросов)

In [63]:
# Возраст куки в днях на момент начала окна наблюдения
train['cookie_age_days'] = (
    (train['window_start_ts'] - train['cookie_created_at']).dt.total_seconds() / 86400
)

# Логарифм возраста куки
train['cookie_age_log'] = np.log1p(train['cookie_age_days'].clip(lower=0))

# Флаг: кука создана внутри окна наблюдения
train['is_created_in_window'] = (
    train['cookie_created_at'] >= train['window_start_ts']
).astype(int)

Временные паттерны:

- `night_activity_ratio` - доля событий ночью 0-5 часов

- `day_activity_ratio` - доля событий днем 6-23 часа

- `hour_std` - стандартное отклонение по часам

In [64]:
events_train['hour'] = events_train['event_ts'].dt.hour
time_features = events_train.groupby('cookie_id').agg(
    night_activity_ratio=('hour', lambda x: ((x >= 0) & (x < 6)).mean()),
    day_activity_ratio=('hour', lambda x: ((x >= 6) & (x < 24)).mean()),
    hour_std=('hour', 'std'),
).reset_index()

Интервалы между событиями:

- `median_interval` - медианный интервал между событиями

- `mean_interval` - средний интервал

- `min_interval` - минимальный интервал

- `ratio_instant_01` - доля событий с интервалом < 0.1 сек 

In [65]:
events_sorted = events_train.sort_values(['cookie_id', 'event_ts'])

# Считаем разницу между текущим и предыдущим событием для каждой куки
events_sorted['time_diff_sec'] = events_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

# Агрегируем интервалы по кукам
interval_features = events_sorted.groupby('cookie_id').agg(
    median_interval=('time_diff_sec', 'median'),
    mean_interval=('time_diff_sec', 'mean'),
    min_interval=('time_diff_sec', 'min'),
    ratio_instant_01=('time_diff_sec', lambda x: (x < 0.1).mean()),
).reset_index()

Теперь объединим все временные фичи:

In [66]:
age_features = train[['cookie_id', 'cookie_age_log', 'is_created_in_window']].copy()

# Объединяем все временные признаки
time_all = age_features.merge(time_features, on='cookie_id', how='left')
time_all = time_all.merge(interval_features, on='cookie_id', how='left')

# Добавляем к основным признакам
base_features = base_features.merge(time_all, on='cookie_id', how='left')

# Заполняем возможные пропуски (например, если у куки только 1 событие — нет интервалов)
base_features = base_features.fillna({
    'hour_std': 0,
    'median_interval': 0,
    'mean_interval': 0,
    'min_interval': 0,
    'ratio_instant_01': 0
})

### **2.2.6. Поисковые признаки**

Признаки:

- `search_query_ratio` - доля событий с поисковым запросом

- `unique_search_queries` - количество уникальных поисковых запросов

- `max_search_page` - максимальная глубина страницы выдачи

- `mean_search_page` - средняя глубина страницы выдачи

In [67]:
search_events = events_train[events_train['search_query'].notna()]

search_all = (
    events_train.groupby('cookie_id')
    .size().reset_index(name='total_events')
    .merge(
        search_events.groupby('cookie_id').agg(
            search_events_count=('search_query', 'size'),
            unique_search_queries=('search_query', 'nunique'),
            max_search_page=('search_page', 'max'),
            mean_search_page=('search_page', 'mean'),
        ).reset_index(),
        on='cookie_id', how='left',
    )
)
search_all[['search_events_count', 'unique_search_queries',
            'max_search_page', 'mean_search_page']] = \
    search_all[['search_events_count', 'unique_search_queries',
                'max_search_page', 'mean_search_page']].fillna(0)
search_all['search_query_ratio'] = search_all['search_events_count'] / search_all['total_events']

base_features = base_features.merge(
    search_all[['cookie_id', 'search_query_ratio', 'unique_search_queries',
                'max_search_page', 'mean_search_page']],
    on='cookie_id', how='left',
)

### **2.2.7. Признаки координат курсора**

- `pointer_ratio` - доля событий с координатами.

- `has_any_pointer` - флаг наличия данных курсора.

- `pointer_x_range`, `pointer_y_range` - диапазон движения курсора.

In [68]:
events_train['has_pointer'] = events_train['pointer_x'].notna() & events_train['pointer_y'].notna()

pointer_features = events_train.groupby('cookie_id').agg(
    pointer_ratio=('has_pointer', 'mean'),
    has_any_pointer=('has_pointer', 'any'),
    pointer_x_range=('pointer_x', np.ptp),
    pointer_y_range=('pointer_y', np.ptp),
).reset_index()

base_features = base_features.merge(pointer_features, on='cookie_id', how='left')

base_features[['pointer_ratio', 'pointer_x_range', 'pointer_y_range']] = \
    base_features[['pointer_ratio', 'pointer_x_range', 'pointer_y_range']].fillna(0.0)
base_features['has_any_pointer'] = base_features['has_any_pointer'].fillna(False).astype(int)

### **2.2.8. Признаки типов событий**

Доли типов событий:

- `ratio_item_view`, `ratio_photo_swipe`, `ratio_favorite_add`, `ratio_login`, `ratio_search_results_view`, `ratio_contact_phone_show`, `ratio_contact_chat_open`, `ratio_contact_message_sent`, `ratio_seller_page_view`, `ratio_captcha_shown`

Бинарные флаги:

- `has_favorite_add` - добавлял ли в избранное

- `has_login` - логинился ли

- `has_captcha` - была ли показана капча

Отношения:

- `item_view_to_photo_ratio` - отношение просмотров к листанию фото

In [69]:
# Доли типов событий для каждой куки
event_type_counts = (
    events_train.groupby('cookie_id')['event_name']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .add_prefix('ratio_')
    .reset_index()
)

# Флаги для важных событий
flags = events_train.assign(
    has_favorite_add = events_train['event_name'] == 'favorite_add',
    has_login        = events_train['event_name'] == 'login',
    has_captcha      = events_train['event_name'] == 'captcha_shown',
).groupby('cookie_id')[['has_favorite_add', 'has_login', 'has_captcha']].any().astype(int).reset_index()

# Отношение item_view / photo_swipe
item_view   = events_train[events_train['event_name'] == 'item_view'].groupby('cookie_id').size()
photo_swipe = events_train[events_train['event_name'] == 'photo_swipe'].groupby('cookie_id').size()

ratio = (item_view / (photo_swipe + 1)).fillna(0).reset_index(name='item_view_to_photo_ratio')

# Собираем всё вместе
event_features = event_type_counts.merge(flags, on='cookie_id', how='left')
event_features = event_features.merge(ratio, on='cookie_id', how='left')

base_features = base_features.merge(event_features, on='cookie_id', how='left')

In [70]:
print(f"Всего признаков: {len(base_features.columns) - 2}")  # минус cookie_id и target

Всего признаков: 46
